In [1]:
%%bash
cd /kaggle/working && rm -rf SGG-Benchmark
git clone -q https://github.com/Maelic/SGG-Benchmark.git
cd SGG-Benchmark && pip install -e . -q
pip install -q ultralytics hydra-core omegaconf
echo "INSTALL DONE"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.8/155.8 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 5.0 MB/s eta 0:00:00
INSTALL DONE


In [2]:
import pathlib
root = pathlib.Path("/kaggle/working/SGG-Benchmark/sgg_benchmark")
old = "from ultralytics.utils.plotting import feature_visualization"
marker = "feature_visualization = None  # removed in newer ultralytics"
NL = chr(10)
new = NL.join([
    "try:",
    "    from ultralytics.utils.plotting import feature_visualization",
    "except ImportError:",
    "    feature_visualization = None  # removed in newer ultralytics; only",
    "    # used by an optional debug path this run never enables",
])

patched = []
for f in root.rglob("*.py"):
    src = f.read_text(encoding="utf-8")
    if marker in src:
        continue
    if old in src:
        f.write_text(src.replace(old, new, 1), encoding="utf-8")
        patched.append(str(f.relative_to(root)))
print(f"patched {len(patched)} file(s):", patched)

import os, subprocess
os.chdir("/kaggle/working/SGG-Benchmark")
r = subprocess.run(["python", "-c",
    "from sgg_benchmark.modeling.detector import build_detection_model; print('IMPORT OK')"],
    capture_output=True, text=True)
print(r.stdout.strip() or r.stderr[-1200:])
assert "IMPORT OK" in r.stdout, "import chain broken — paste the error above"

patched 2 file(s): ['modeling/backbone/yoloe.py', 'modeling/backbone/yolo.py']
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
IMPORT OK


In [3]:
import os, glob, json, shutil
os.chdir("/kaggle/working/SGG-Benchmark")

yaml_hit = glob.glob("/kaggle/input/**/spatial_sgg_react.yaml", recursive=True)
assert yaml_hit, "base dataset not attached"
INPUT = os.path.dirname(yaml_hit[0])

auto = {}
for split in ("train", "val"):
    hits = [p for p in glob.glob(f"/kaggle/input/**/*{split}*annotations.auto.coco.json",
                                 recursive=True) if "auto-085" in p]
    assert hits, f"new {split} auto labels not found — is spatial-sgg-auto-085 attached?"
    auto[split] = hits[0]

print("base data :", INPUT)
for k, v in auto.items():
    print(f"new {k:5}:", v)

os.makedirs("datasets", exist_ok=True)
os.makedirs("configs/hydra/Spatial", exist_ok=True)
for d in ("spatial_sgg", "spatial_sgg_yolo"):
    if not os.path.isdir(f"datasets/{d}"):
        shutil.copytree(f"{INPUT}/{d}", f"datasets/{d}")
shutil.copy(f"{INPUT}/spatial_sgg_react.yaml", "configs/hydra/Spatial/react.yaml")

for split, src in auto.items():
    shutil.copy(src, f"datasets/spatial_sgg/{split}/_annotations.auto.coco.json")

for split, want in (("train", 121492), ("val", 18124)):
    d = json.load(open(f"datasets/spatial_sgg/{split}/_annotations.auto.coco.json"))
    n = len(d["rel_annotations"])
    bg = d["categories"][0]["name"] == "__background__"
    nr = d["rel_categories"][0]["name"] == "__no_relation__"
    print(f"{split}: {n} relations (expect {want}), bg0={bg}, norel0={nr}")
    assert n == want, f"{split}: got {n}, expected {want} — OLD LABELS, stop"
    assert bg and nr, f"{split}: background patch missing — training would give mR=0"

print("\nVERIFIED — new 0.85 labels staged, patch intact")

base data : /kaggle/input/datasets/shah9212/spatial-sgg
new train: /kaggle/input/datasets/shah9212/spatial-sgg-auto-085/train_annotations.auto.coco.json
new val  : /kaggle/input/datasets/shah9212/spatial-sgg-auto-085/val_annotations.auto.coco.json
train: 121492 relations (expect 121492), bg0=True, norel0=True
val: 18124 relations (expect 18124), bg0=True, norel0=True

VERIFIED — new 0.85 labels staged, patch intact


In [4]:
import os, glob, shutil, yaml
os.chdir("/kaggle/working/SGG-Benchmark")
os.makedirs("checkpoints/BACKBONES", exist_ok=True)

found = glob.glob("/kaggle/input/**/yolov8m_spatial.pt", recursive=True)
if found:
    shutil.copy(found[0], "checkpoints/BACKBONES/yolov8m_spatial.pt")
    print("detector reused from", found[0])
else:
    print("no detector in input — training (~15 min)")
    yp = "datasets/spatial_sgg_yolo/data.yaml"
    d = yaml.safe_load(open(yp))
    d["path"] = os.path.abspath("datasets/spatial_sgg_yolo")
    yaml.safe_dump(d, open(yp, "w"))
    from ultralytics import YOLO
    YOLO("yolov8m.pt").train(data=yp, epochs=60, imgsz=640, batch=16,
                             project="det", name="yolov8m_spatial", verbose=False)
    src = max(glob.glob("runs/detect/det/yolov8m_spatial*/weights/best.pt"),
              key=os.path.getmtime)
    shutil.copy(src, "checkpoints/BACKBONES/yolov8m_spatial.pt")
    print("DETECTOR DONE ->", src)

no detector in input — training (~15 min)
Ultralytics 8.4.121 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=datasets/spatial_sgg_yolo/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_

In [5]:
import subprocess, shutil, os, re, time
os.chdir("/kaggle/working/SGG-Benchmark")

def stage():
    """Auto labels for train/val; test is ALWAYS human gold — the design guarantee."""
    for split in ("train", "val"):
        shutil.copy(f"datasets/spatial_sgg/{split}/_annotations.auto.coco.json",
                    f"datasets/spatial_sgg/{split}/_annotations.coco.json")
    shutil.copy("datasets/spatial_sgg/test/_annotations.human.coco.json",
                "datasets/spatial_sgg/test/_annotations.coco.json")

results = {}
for seed in (42, 43, 44):
    tag = f"react_auto_s{seed}"
    stage()
    os.system(f"rm -rf checkpoints/spatial/{tag}")
    cmd = ("python -u tools/relation_train_net_hydra.py "
           "--config-path ../configs/hydra/Spatial --config-name react "
           f"--task sgdet --save-best seed={seed} "
           f"output_dir=./checkpoints/spatial/{tag}")
    t0 = time.time()
    print("=" * 78, f"\nTRAIN {tag}\n", "=" * 78, flush=True)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = r.stdout + "\n" + r.stderr
    print(out[-1500:], flush=True)
    mrs = [float(x) for x in re.findall(r"Result for mR:\s*([\d.]+)", out)]
    results[tag] = max(mrs) if mrs else 0.0
    print(f"\n>>> {tag}: val mR={results[tag]:.4f}  ({(time.time()-t0)/60:.1f} min)\n", flush=True)
    assert results[tag] > 0, f"{tag} produced mR=0 — stop and report"

print("TRAINING SUMMARY:", results)

TRAIN react_auto_s42
| 65/100 [00:03<00:01, 20.76it/s]
100%|██████████| 100/100 [00:04<00:00, 20.19it/s]

SGG Eval: 100%|██████████| 100/100 [00:01<00:00, 62.19it/s]


>>> react_auto_s42: val mR=0.1876  (30.6 min)

TRAIN react_auto_s43
67/100 [00:03<00:01, 20.70it/s]
100%|██████████| 100/100 [00:04<00:00, 20.51it/s]

SGG Eval: 100%|██████████| 100/100 [00:01<00:00, 68.12it/s]


>>> react_auto_s43: val mR=0.1836  (26.9 min)

TRAIN react_auto_s44
| 66/100 [00:03<00:01, 20.41it/s]
100%|██████████| 100/100 [00:04<00:00, 20.07it/s]

SGG Eval: 100%|██████████| 100/100 [00:01<00:00, 63.29it/s]


>>> react_auto_s44: val mR=0.1857  (26.8 min)

TRAINING SUMMARY: {'react_auto_s42': 0.1876, 'react_auto_s43': 0.1836, 'react_auto_s44': 0.1857}


In [6]:
import json, os, shutil
os.chdir("/kaggle/working/SGG-Benchmark")

# fixed zero-shot reference: the HUMAN training annotation
for split in ["train", "val"]:
    shutil.copy(f"datasets/spatial_sgg/{split}/_annotations.human.coco.json",
                f"datasets/spatial_sgg/{split}/_annotations.coco.json")
print("train/val staged to HUMAN (defines the seen-triplet set; expect 94)")

TEST = "datasets/spatial_sgg/test"
full = json.load(open(f"{TEST}/_annotations.human.coco.json"))
shutil.copy(f"{TEST}/_annotations.human.coco.json", f"{TEST}/_annotations.full.coco.json")

def subset(group):
    keep = {im["id"] for im in full["images"] if im["file_name"].startswith(group + "_")}
    ann  = [a for a in full["annotations"] if a["image_id"] in keep]
    akeep = {a["id"] for a in ann}
    rel  = [r for r in full["rel_annotations"]
            if r["subject_id"] in akeep and r["object_id"] in akeep]
    d = dict(full)
    d["images"]          = [im for im in full["images"] if im["id"] in keep]
    d["annotations"]     = ann
    d["rel_annotations"] = rel
    path = f"{TEST}/_annotations.{group}.coco.json"
    json.dump(d, open(path, "w"))
    print(f"  {group}: {len(d['images'])} images, {len(rel)} relations")
    return path

SLICES = {"full": f"{TEST}/_annotations.full.coco.json"}
for g in ["group_6", "group_7", "group_8"]:
    SLICES[g] = subset(g)

train/val staged to HUMAN (defines the seen-triplet set; expect 94)
  group_6: 100 images, 970 relations
  group_7: 99 images, 796 relations
  group_8: 37 images, 1052 relations


In [7]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import sys, json, glob, shutil, gc, torch, logging, statistics as st
os.chdir("/kaggle/working/SGG-Benchmark")

RUNS = {}
for seed in (42, 43, 44):
    tag = f"react_auto_s{seed}"
    ck = sorted(glob.glob(f"checkpoints/spatial/{tag}/*.pth"))
    assert ck, f"no checkpoint for {tag} — did training finish?"
    RUNS[tag] = {"arm": "auto", "seed": seed,
                 "cfg": f"checkpoints/spatial/{tag}/config.yml", "ckpt": ck[-1]}
print("runs:", {k: os.path.basename(v["ckpt"]) for k, v in RUNS.items()})

for _m in [m for m in list(sys.modules) if m.startswith("sgg_benchmark")]:
    del sys.modules[_m]

from omegaconf import OmegaConf
from sgg_benchmark.modeling.detector import build_detection_model
from sgg_benchmark.utils.checkpoint import DetectronCheckpointer
from sgg_benchmark.data import make_data_loader
from sgg_benchmark.engine.inference import inference
try:
    from sgg_benchmark.utils.logger import setup_logger
    logger = setup_logger("sgg_benchmark", ".", 0, verbose="INFO", steps=True)
except Exception:
    logging.basicConfig(level=logging.INFO); logger = logging.getLogger("sgg_benchmark")

TEST = "datasets/spatial_sgg/test"
OUTJSON = "/kaggle/working/reeval_auto_085.json"
RESULTS = json.load(open(OUTJSON)) if os.path.exists(OUTJSON) else {}
print(f"resuming with {len(RESULTS)} result(s) already recorded")

def harvest(out, name, info, slice_name):
    f = f"{out}/eval_results_top_100.json"
    if not os.path.exists(f):
        return False
    d = json.load(open(f))
    zs = d.get("sgdet_zeroshot_recall", {}).get("100", [])
    RESULTS[f"{name}|{slice_name}"] = {
        "arm": "auto", "seed": info["seed"], "slice": slice_name,
        "R@100":  st.mean(d["sgdet_recall"]["100"]) if d.get("sgdet_recall") else None,
        "mR@100": d.get("sgdet_mean_recall", {}).get("100"),
        "F1@100": d.get("sgdet_f1_score", {}).get("100"),
        "zR@100": (st.mean(zs) if zs else 0.0),
        "n_zeroshot": len(zs),
    }
    json.dump(RESULTS, open(OUTJSON, "w"), indent=1)   # checkpoint after every slice
    return True

for name, info in RUNS.items():
    todo = [s for s in SLICES if f"{name}|{s}" not in RESULTS]
    if not todo:
        print(f"SKIP {name} — all slices done"); continue

    cfg = OmegaConf.load(info["cfg"])
    model = build_detection_model(cfg).to(cfg.model.device)
    DetectronCheckpointer(cfg, model).load(info["ckpt"])
    model.eval()

    for slice_name in todo:
        out = f"./checkpoints/spatial/re_{name}_{slice_name}"
        if harvest(out, name, info, slice_name):
            print(f"SKIP {name}|{slice_name} — recovered from disk"); continue
        shutil.copy(SLICES[slice_name], f"{TEST}/_annotations.coco.json")
        os.makedirs(out, exist_ok=True)
        cfg.output_dir = out
        loader = make_data_loader(cfg, mode="test")[0]
        print("=" * 78, f"\nEVAL {name} on {slice_name} ({len(loader.dataset)} images)", flush=True)
        with torch.no_grad():
            inference(cfg, model, loader, dataset_name="SpatialRobot_test",
                      iou_types=("bbox", "relations"), box_only=False,
                      device=cfg.model.device, expected_results=[],
                      expected_results_sigma_tol=4, output_folder=out, logger=logger)
        harvest(out, name, info, slice_name)
        del loader
        gc.collect(); torch.cuda.empty_cache()
        print(f"  GPU after {slice_name}: {torch.cuda.memory_allocated()/1e9:.2f} GB", flush=True)

    del model
    gc.collect(); torch.cuda.empty_cache()
    print(f"FREED {name}; GPU now {torch.cuda.memory_allocated()/1e9:.2f} GB\n", flush=True)

print("\n===== RESULTS =====")
print(json.dumps(RESULTS, indent=1))
print(f"\n{len(RESULTS)}/{len(RUNS)*len(SLICES)} evaluations complete")

runs: {'react_auto_s42': 'best_model_epoch_21.pth', 'react_auto_s43': 'best_model_epoch_19.pth', 'react_auto_s44': 'best_model_epoch_24.pth'}
resuming with 0 result(s) already recorded
Overriding model.yaml nc=80 with nc=6

                   from  n    params  module                                       arguments                     
  0                  -1  1      1392  ultralytics.nn.modules.conv.Conv             [3, 48, 3, 2]                 
  1                  -1  1     41664  ultralytics.nn.modules.conv.Conv             [48, 96, 3, 2]                
  2                  -1  2    111360  ultralytics.nn.modules.block.C2f             [96, 96, 2, True]             
  3                  -1  1    166272  ultralytics.nn.modules.conv.Conv             [96, 192, 3, 2]               
  4                  -1  4    813312  ultralytics.nn.modules.block.C2f             [192, 192, 4, True]           
  5                  -1  1    664320  ultralytics.nn.modules.conv.Conv             [192, 384

100%|██████████| 210/210 [00:09<00:00, 21.73it/s]

2026-08-18 18:00:14,120 sgg_benchmark INFO: Total run time: 0:00:09 (43.019481440952845 ms / img per device, on 1 devices)
2026-08-18 18:00:14,121 sgg_benchmark INFO: Average latency per image: 43.019481440952845ms
2026-08-18 18:00:14,122 sgg_benchmark INFO: Standard deviation of latency: 10.475395273231749ms


2026-08-18 18:00:14,191 sgg_benchmark.data.build INFO: ----------------------------------------------------------------------------------------------------
2026-08-18 18:00:14,191 sgg_benchmark.data.build INFO: get dataset statistics...
2026-08-18 18:00:14,192 sgg_benchmark.data.build INFO: Unable to load data statistics from: ./checkpoints/spatial/re_react_auto_s42_full/SpatialRobot_statistics.cache
2026-08-18 18:00:14,293 sgg_benchmark.data.build INFO: Save data statistics to: ./checkpoints/spatial/re_react_auto_s42_full/SpatialRobot_statistics.cache
2026-08-18 18:00:14,293 sgg_benchmark.data.build INFO: ----------------------------------------------------------------------------------------------------
2026-08-18 18:00:14,300 sgg_benchmark INFO: Dynamically loaded 94 seen triplets from training statistics for zero-shot evaluation.
creating index...
index created!
Loading and preparing results...
Converting ndarray to lists...
(14022, 7)
0/14022
DONE (t=0.04s)
creating index...
index

SGG Eval: 100%|██████████| 210/210 [00:01<00:00, 156.48it/s]

...

2026-08-18 18:01:38,360 sgg_benchmark INFO: Total run time: 0:00:01 (46.947887729954076 ms / img per device, on 1 devices)
2026-08-18 18:01:38,361 sgg_benchmark INFO: Average latency per image: 46.947887729954076ms
2026-08-18 18:01:38,362 sgg_benchmark INFO: Standard deviation of latency: 4.624231281126217ms
2026-08-18 18:01:38,379 sgg_benchmark.data.build INFO: ----------------------------------------------------------------------------------------------------
2026-08-18 18:01:38,380 sgg_benchmark.data.build INFO: get dataset statistics...
2026-08-18 18:01:38,381 sgg_benchmark.data.build INFO: Unable to load data statistics from: ./checkpoints/spatial/re_react_auto_s44_group_8/SpatialRobot_statistics.cache


2026-08-18 18:01:38,480 sgg_benchmark.data.build INFO: Save data statistics to: ./checkpoints/spatial/re_react_auto_s44_group_8/SpatialRobot_statistics.cache
2026-08-18 18:01:38,481 sgg_benchmark.data.build INFO: ----------------------------------------------------------------------------------------------------
2026-08-18 18:01:38,487 sgg_benchmark INFO: Dynamically loaded 94 seen triplets from training statistics for zero-shot evaluation.
creating index...
index created!
Loading and preparing results...
Converting ndarray to lists...
(2921, 7)
0/2921
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.25s).
Accumulating evaluation results...
DONE (t=0.05s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.323
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.691
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.271
 Average Precision  (A

SGG Eval: 100%|██████████| 37/37 [00:00<00:00, 124.80it/s]

2026-08-18 18:01:39,134 sgg_benchmark INFO: 
Detection evaluation mAp=0.6913
SGG eval:     R @ 20: 0.0230;     R @ 50: 0.0541;     R @ 100: 0.0862;  for mode=sgdet.
SGG eval:    mR @ 20: 0.0363;    mR @ 50: 0.0812;    mR @ 100: 0.1223;  for mode=sgdet.
----------------------- Details ------------------------
(on:0.0000) (under:0.5721) (to the left of:0.0693) (to the right of:0.0786) (in front of:0.0030) (behind:0.0383) (near:0.0946) 
--------------------------------------------------------
SGG eval:     zR @ 20: 0.0134;     zR @ 50: 0.0274;     zR @ 100: 0.0468;  for mode=sgdet.
SGG eval:     F1 @ 20: 0.0282;     F1 @ 50: 0.0650;     F1 @ 100: 0.1011;  for mode=sgdet.



  GPU after group_8: 0.18 GB
FREED react_auto_s44; GPU now 0.03 GB


===== RESULTS =====
{
 "react_auto_s42|full": {
  "arm": "auto",
  "seed": 42,
  "slice": "full",
  "R@100": 0.26029777041594176,
  "mR@100": 0.2901552881480745,
  "F1@100": 0.2744167678034815,
  "zR@100": 0.2946666666666667,
  "n_zeroshot": 125
 },
 "react_auto_s42|group_6": {
  "arm": "auto",
  "seed": 42,
  "slice": "group_6",
  "R@100": 0.3093160173160173,
  "mR@100": 0.30957469459932513,
  "F1@100": 0.30944530189809694,
  "zR@100": 0.42424242424242425,
  "n_zeroshot": 33
 },
 "react_auto_s42|group_7": {
  "arm": "auto",
  "seed": 42,
  "slice": "group_7",
  "R@100": 0.280876274568225,
  "mR@100": 0.2811907052314319,
  "F1@100": 0.28103340195066445,
  "zR@100": 0.3463541666666667,
  "n_zeroshot": 64
 },
 "react_auto_s42|group_8": {
  "arm": "auto",
  "seed": 42,
  "slice": "group_8",
  "R@100": 0.08577143807694092,
  "mR@100": 0.1166161219732648,
  "F1@100": 0.09884335264599879,
  "zR@100": 0.023809523809523808,
 

In [8]:
%%bash
cd /kaggle/working/SGG-Benchmark
zip -rq /kaggle/working/auto_085_results.zip checkpoints/spatial -x "*.pth" -x "*.pt"
ls -la /kaggle/working/auto_085_results.zip /kaggle/working/reeval_auto_085.json

-rw-r--r-- 1 root root 207293 Aug 18 18:01 /kaggle/working/auto_085_results.zip
-rw-r--r-- 1 root root   2782 Aug 18 18:01 /kaggle/working/reeval_auto_085.json
